# 05 Fine-tuned text model (optional, added after the first test evaluation)

- What it does: fine-tunes the text encoder on inspector comments, adds its score to the history features (model M3), and compares it with M1, M2 and a bag of words.
- Inputs: `data/processed/model_table.parquet`, `data/processed/inspections_full.parquet`, the predictions in `preds/`, and `docs/FROZEN_CONFIG_M3.json` for the test part.
- Outputs: `preds/M3_lgbm_finetuned_val.csv`, `preds/M3_lgbm_finetuned_test.csv`, `results/finetune_val.csv`, `results/finetune_test.csv`, `results/m3_selected_settings.json`.

This notebook is not part of the 01 to 04 chain. It trains nine neural networks (about 40 minutes on an Apple-silicon GPU, much longer on a CPU), and neural-network training can differ slightly between runs and machines, so its numbers are not guaranteed to reproduce to the last digit. The predictions from our run are saved in `preds/`.

M3 was built **after** the five planned models had been scored on the test years. Its design and settings are chosen on the validation year only and frozen before its own test scoring, but the decision to build it came after seeing the first test results. The final model of the project remains M1.

In [1]:
import sys
sys.path.insert(0, "..")

import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from transformers import AutoTokenizer

warnings.filterwarnings("ignore")

from src import evaluate as E
from src import features as F
from src import finetune as FT
from src import impact as I
from src import models as M
from src import text as T

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 200)

## 1. What the text model learns from

One training example = one routine inspection that has comments from an earlier visit. The input is those comments, and the target is whether this inspection failed. Inspections before 2016 are used too: they are not in the model table, but they are valid past examples for the text model.

We try two text sources. Notebook 03 showed that they are the same words for 9 rows in 10, yet text-only models do better on the second:

- `previous_visit`: the comments of the previous visit (what M2 uses);
- `last_routine`: the comments of the last routine inspection.

In [2]:
SOURCES = {"previous_visit": "prior_comments", "last_routine": "last_routine_comments"}

table = M.load_model_table("../data/processed/model_table.parquet")
full = pd.read_parquet("../data/processed/inspections_full.parquet",
                       columns=["inspection_id", "licenseno", "date", "label_fail", "is_reinspection", "comments", "prior_comments"])
# last_routine_comments only looks back (shift then carry forward), so computing it on all rows leaks nothing.
full = full.merge(F.candidate_last_routine_comments(full), on="inspection_id")
table = table.merge(full[["inspection_id", "last_routine_comments"]], on="inspection_id", how="left")
routine = full[full["is_reinspection"] == 0].sort_values("date").reset_index(drop=True)


def pairs_for(text_col):
    """Routine inspections that have text in `text_col`."""
    return routine.loc[routine[text_col].str.strip() != "", ["inspection_id", "date", text_col, "label_fail"]]


for name, col in SOURCES.items():
    p = pairs_for(col)
    print(f"{name:15s} {int((p['date'] < '2024-01-01').sum()):,} examples before 2024")
print("settings:", {"max_tokens": FT.MAX_TOKENS, "epochs": FT.EPOCHS, "learning_rate": FT.LEARNING_RATE, "batch_size": FT.BATCH_SIZE})

previous_visit  53,742 examples before 2024
last_routine    54,013 examples before 2024
settings: {'max_tokens': 128, 'epochs': 2, 'learning_rate': 5e-05, 'batch_size': 32}


## 2. Text scores that never look ahead

If one text model were trained on 2016 to 2022 and then scored those same years, its scores there would be too good, because it has seen the answers. LightGBM would then trust the text score more than it deserves.

So we train in time blocks. Each model is trained on everything before a cutoff and scores only the inspections after it. The last block scores the validation year with a model trained on everything before 2023.

In [3]:
FT.self_test()
scores_val = {}
for name, col in SOURCES.items():
    before_test = pairs_for(col)
    before_test = before_test[before_test["date"] < "2024-01-01"]          # all that the validation part may see
    scores_val[name] = FT.forward_text_scores(before_test, FT.BLOCKS_VALIDATION, text_col=col, name=name,
                                              cache_dir="../data/processed")

W1111 21:40:13.639000 92432 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


finetune.py self-test passed: P(fail | 'soiled') = 0.94, P(fail | 'clean') = 0.09
  loaded cached text scores (previous_visit) for 2016 to 2019
  loaded cached text scores (previous_visit) for 2019 to 2022
  loaded cached text scores (previous_visit) for 2022 to 2023
  loaded cached text scores (previous_visit) for 2023 to 2024
  loaded cached text scores (last_routine) for 2016 to 2019
  loaded cached text scores (last_routine) for 2019 to 2022
  loaded cached text scores (last_routine) for 2022 to 2023
  loaded cached text scores (last_routine) for 2023 to 2024


In [4]:
TEXT_FT = ["text_score", "has_text"]
FEATURES_M3 = M.HISTORY_FEATURES + TEXT_FT
is_train = (table["split"] == "train").to_numpy()
is_val = (table["split"] == "val").to_numpy()
y_val, val_groups = table.loc[is_val, M.TARGET].to_numpy(), table.loc[is_val, "licenseno"].to_numpy()


def with_text_score(table, scores, text_col):
    """Add the fine-tuned text score (missing when there is no text) and a has-text flag."""
    out = table.merge(scores, on="inspection_id", how="left")
    assert (out["inspection_id"].to_numpy() == table["inspection_id"].to_numpy()).all()
    out["has_text"] = (out[text_col].str.strip() != "").astype(int)
    return out


tables_val = {name: with_text_score(table, scores_val[name], col) for name, col in SOURCES.items()}
for name, t in tables_val.items():
    v = t[is_val]
    covered = v["text_score"].notna().to_numpy()
    assert (covered == (v["has_text"] == 1).to_numpy()).all(), "every row with text must have a text score"
    print(f"{name:15s} validation rows with a text score: {covered.mean():.1%} | "
          f"AUC of the raw score on those rows: {E.evaluate(y_val[covered], v.loc[covered, 'text_score'])['auc']:.3f}")

previous_visit  validation rows with a text score: 66.9% | AUC of the raw score on those rows: 0.680
last_routine    validation rows with a text score: 66.9% | AUC of the raw score on those rows: 0.702


## 3. Validation (2023)

For each text source, a text-only model: the fine-tuned score and the has-text flag, with no history feature. It is tuned with the same grid as M1 and M2.

The rule, set before looking: M3 uses the text source whose text-only model has the higher validation hit rate.

In [5]:
text_only = {}
for name, t in tables_val.items():
    _, params = M.grid_search_lgbm(t[is_train], t[is_val], TEXT_FT)
    score = M.fit_lgbm(t[is_train], TEXT_FT, params).predict_proba(t[is_val][TEXT_FT])[:, 1]
    text_only[name] = {"params": params, "score": score, "precision_at_k": E.evaluate(y_val, score)["precision_at_k"]}
    print(f"text only, fine-tuned on {name:15s} hit rate {text_only[name]['precision_at_k']:.1%}")
CHOSEN = max(text_only, key=lambda n: text_only[n]["precision_at_k"])
print("text source for M3:", CHOSEN)

text only, fine-tuned on previous_visit  hit rate 85.2%


text only, fine-tuned on last_routine    hit rate 86.9%
text source for M3: last_routine


In [6]:
train, val = tables_val[CHOSEN][is_train], tables_val[CHOSEN][is_val]
grid_m3, params_m3 = M.grid_search_lgbm(train, val, FEATURES_M3)
m3 = M.fit_lgbm(train, FEATURES_M3, params_m3)
score_m3 = m3.predict_proba(val[FEATURES_M3])[:, 1]
M.save_preds(val, score_m3, "../preds/M3_lgbm_finetuned_val.csv")
print("selected M3 settings:", params_m3)
print(M.importance_table(m3, FEATURES_M3, top=6).round(3).to_string(index=False))

selected M3 settings: {'num_leaves': 7, 'learning_rate': 0.05, 'n_estimators': 100}
                feature      gain  share_of_gain
             text_score 34874.517          0.369
    last_routine_n_viol 14842.645          0.157
prior_routine_fail_rate 12864.986          0.136
        prior_fail_rate  8393.754          0.089
                    zip  8160.571          0.086
              last_fail  7082.341          0.075


In [7]:
score_m1 = pd.read_csv("../preds/M1_lgbm_val.csv")["score"].to_numpy()
score_m2 = pd.read_csv("../preds/M2_lgbm_text_val.csv")["score"].to_numpy()
candidates = {"M1: history only": score_m1,
              "M2: history + frozen embeddings": score_m2,
              f"M3: history + fine-tuned encoder ({CHOSEN})": score_m3}
candidates.update({f"text only: fine-tuned encoder, {name}": r["score"] for name, r in text_only.items()})
rows = []
for name, score in candidates.items():
    row = {"model": name, **{k: E.evaluate(y_val, score)[k] for k in ["precision_at_k", "auc"]}}
    if name != "M1: history only":
        d, lo, hi = E.paired_differences(y_val, score, score_m1, groups=val_groups)["precision_at_k"]
        row.update({"minus_M1": d, "ci_lo": lo, "ci_hi": hi, "verdict": E.verdict(d, lo, hi)})
    rows.append(row)
finetune_val = pd.DataFrame(rows)

# For comparison: text-only rows from notebook 03 (same year, same method, frozen encoder and bag of words).
earlier = pd.read_csv("../results/text_ablation_val.csv")
finetune_val = pd.concat([finetune_val, earlier[earlier["model"].str.startswith("text only")]], ignore_index=True)
finetune_val.to_csv("../results/finetune_val.csv", index=False)
finetune_val.round(3)

,model,precision_at_k,auc,minus_M1,ci_lo,ci_hi,verdict
0,M1: history only,0.905,0.780,NaN,NaN,NaN,NaN
1,M2: history + frozen embeddings,0.906,0.783,0.001,-0.012,0.010,within noise
2,M3: history + fine-tuned encoder (last_routine),0.913,0.785,0.008,-0.004,0.017,within noise
3,"text only: fine-tuned encoder, previous_visit",0.852,0.729,-0.053,-0.074,-0.032,beyond noise
4,"text only: fine-tuned encoder, last_routine",0.869,0.739,-0.036,-0.054,-0.016,beyond noise
5,"text only: embeddings, previous visit",0.839,0.724,-0.066,-0.088,-0.043,beyond noise
6,"text only: embeddings, last routine inspection",0.855,0.736,-0.050,-0.071,-0.029,beyond noise
7,"text only: bag of words, last routine inspection",0.871,0.738,-0.034,-0.055,-0.017,beyond noise


### Like for like: fine-tuned encoder against a bag of words on the same text

The fine-tuned encoder reads the first 128 tokens of a comment, and a bag of words can read all of it. Here the bag of words is built on each text source twice, on the whole comment and on its first 128 tokens, and compared with the fine-tuned encoder on that same source. All rows are text-only models.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(FT.MODEL_NAME, revision=FT.MODEL_REVISION)
rows = []
for name, col in SOURCES.items():
    texts = table[col].fillna("")
    has = (texts.str.strip() != "").to_numpy()
    token_ids = tokenizer(texts[has].tolist())["input_ids"]
    first_tokens = texts.copy()
    first_tokens[has] = tokenizer.batch_decode([t[:FT.MAX_TOKENS] for t in token_ids], skip_special_tokens=True)
    rows.append({"text source": name, "model": "fine-tuned encoder (first 128 tokens)",
                 "precision_at_k": text_only[name]["precision_at_k"],
                 "comments longer than 128 tokens": np.mean([len(t) > FT.MAX_TOKENS for t in token_ids])})
    for label, variant in [("bag of words, whole comment", texts), ("bag of words, first 128 tokens", first_tokens)]:
        block = pd.DataFrame(T.bag_of_words_features(variant, is_train & has), columns=T.PC_COLUMNS,
                             index=table.index).assign(has_text=has.astype(int))
        df = pd.concat([table[[M.TARGET]], block], axis=1)
        _, best = M.grid_search_lgbm(df[is_train], df[is_val], list(block.columns))
        score = M.fit_lgbm(df[is_train], list(block.columns), best).predict_proba(df[is_val][list(block.columns)])[:, 1]
        rows.append({"text source": name, "model": label, "precision_at_k": E.evaluate(y_val, score)["precision_at_k"]})
like_for_like = pd.DataFrame(rows)
like_for_like.to_csv("../results/finetune_like_for_like_val.csv", index=False)
like_for_like.round(3)

,text source,model,precision_at_k,comments longer than 128 tokens
0,previous_visit,fine-tuned encoder (first 128 tokens),0.852,0.268
1,previous_visit,"bag of words, whole comment",0.841,NaN
2,previous_visit,"bag of words, first 128 tokens",0.832,NaN
3,last_routine,fine-tuned encoder (first 128 tokens),0.869,0.322
4,last_routine,"bag of words, whole comment",0.871,NaN
5,last_routine,"bag of words, first 128 tokens",0.861,NaN


In [9]:
selected = {
    "model": "M3_lgbm_finetuned",
    "status": "added after the first test evaluation; chosen on validation (2023) only; the project's final model stays M1",
    "text_source": CHOSEN,
    "text_source_rule": "the source whose text-only fine-tuned model has the higher validation hit rate",
    "features": "history + text_score (fine-tuned encoder) + has_text",
    "lgbm": params_m3,
    "text_only_lgbm": text_only[CHOSEN]["params"],
    "fine_tuning": {"encoder": FT.MODEL_NAME, "encoder_revision": FT.MODEL_REVISION, "max_tokens": FT.MAX_TOKENS,
                    "epochs": FT.EPOCHS, "learning_rate": FT.LEARNING_RATE, "batch_size": FT.BATCH_SIZE,
                    "target": "label_fail of the routine inspection"},
    "time_blocks": {"validation": FT.BLOCKS_VALIDATION, "test": FT.BLOCK_TEST},
    "test_protocol": "refit LightGBM on 2016 to 2023 with these settings; test rows get their text score from a model "
                     "fine-tuned on everything before 2024; scored once; report M3 and the text-only model next to the "
                     "five planned models, with paired differences M3 minus M1, M2 and B2",
}
json.dump(selected, open("../results/m3_selected_settings.json", "w"), indent=2)
print(json.dumps({k: selected[k] for k in ["text_source", "lgbm", "text_only_lgbm"]}))

{"text_source": "last_routine", "lgbm": {"num_leaves": 7, "learning_rate": 0.05, "n_estimators": 100}, "text_only_lgbm": {"num_leaves": 15, "learning_rate": 0.05, "n_estimators": 57}}


## 4. Test years (runs only after the M3 settings are frozen)

This part runs when `docs/FROZEN_CONFIG_M3.json` exists. That file is a copy of the settings above, committed before the test years are scored with M3.

In [10]:
frozen_path = Path("../docs/FROZEN_CONFIG_M3.json")
RUN_TEST = frozen_path.exists()
print("M3 settings frozen:", RUN_TEST)

M3 settings frozen: True


In [11]:
if RUN_TEST:
    FROZEN = json.load(open(frozen_path))
    source, col = FROZEN["text_source"], SOURCES[FROZEN["text_source"]]
    # One more text model, trained on everything before the test period, scores the test years.
    scores_test = FT.forward_text_scores(pairs_for(col), [FT.BLOCK_TEST], text_col=col, name=source, cache_dir="../data/processed")
    table_all = with_text_score(table, pd.concat([scores_val[source], scores_test], ignore_index=True), col)
    is_fit = table_all["split"].isin(["train", "val"]).to_numpy()
    is_test = (table_all["split"] == "test").to_numpy()
    fit, test = table_all[is_fit], table_all[is_test]
    assert fit["date"].max() < pd.Timestamp("2024-01-01") <= test["date"].min()
    assert (test["text_score"].notna() == (test["has_text"] == 1)).all()
    y_test, test_groups = test[M.TARGET].to_numpy(), test["licenseno"].to_numpy()

    test_scores = {name: pd.read_csv(f"../preds/{name}_test.csv")["score"].to_numpy()
                   for name in ["B0_random", "B1_past_fail_rate", "B2_logreg", "M1_lgbm", "M2_lgbm_text"]}
    test_scores["T_finetuned_text_only"] = M.fit_lgbm(fit, TEXT_FT, FROZEN["text_only_lgbm"]).predict_proba(test[TEXT_FT])[:, 1]
    test_scores["M3_lgbm_finetuned"] = M.fit_lgbm(fit, FEATURES_M3, FROZEN["lgbm"]).predict_proba(test[FEATURES_M3])[:, 1]
    M.save_preds(test, test_scores["M3_lgbm_finetuned"], "../preds/M3_lgbm_finetuned_test.csv")

    results = E.results_table(y_test, test_scores, groups=test_groups)
    diffs = E.difference_table(y_test, test_scores, "M3_lgbm_finetuned", ["M1_lgbm", "M2_lgbm_text", "B2_logreg"], groups=test_groups)
    hit = diffs[diffs["metric"] == "precision_at_k"].set_index("minus")
    is_m3 = results["model"] == "M3_lgbm_finetuned"
    for other in hit.index:
        results.loc[is_m3, f"minus_{other}"] = hit.loc[other, "difference"]
        results.loc[is_m3, f"minus_{other}_lo"] = hit.loc[other, "ci_lo"]
        results.loc[is_m3, f"minus_{other}_hi"] = hit.loc[other, "ci_hi"]
        results.loc[is_m3, f"verdict_vs_{other}"] = hit.loc[other, "verdict"]
    # Share of all serious (level 3) violations found in the first 20% of visits: an outcome measure, as in notebook 04.
    results["serious_violations_share"] = [I.severity_in_first_visits(test["n_lvl3"].to_numpy(), s)[0] for s in test_scores.values()]
    results.to_csv("../results/finetune_test.csv", index=False)
    print(results[["model", "auc", "catch_at_k", "precision_at_k", "precision_at_k_lo", "precision_at_k_hi",
                   "serious_violations_share"]].round(3).to_string(index=False))
    print()
    for other, r in hit.iterrows():
        print(f"M3 minus {other}: {r['difference'] * 100:+.1f} points of hit rate "
              f"(95% CI {r['ci_lo'] * 100:+.1f} to {r['ci_hi'] * 100:+.1f}): {r['verdict']}")
    by_year = {int(y): round(float(E.evaluate(y_test[idx], test_scores["M3_lgbm_finetuned"][idx])["precision_at_k"]), 3)
               for y, idx in test.reset_index(drop=True).groupby("year").indices.items()}
    print("\nM3 hit rate by year:", by_year)
else:
    print("Skipped: freeze the M3 settings first (copy results/m3_selected_settings.json to docs/FROZEN_CONFIG_M3.json and commit).")

  loaded cached text scores (last_routine) for 2024 to 2100


                model   auc  catch_at_k  precision_at_k  precision_at_k_lo  precision_at_k_hi  serious_violations_share
            B0_random 0.500       0.200           0.590              0.578              0.599                     0.200
    B1_past_fail_rate 0.698       0.272           0.803              0.785              0.817                     0.332
            B2_logreg 0.736       0.295           0.870              0.852              0.882                     0.388
              M1_lgbm 0.748       0.299           0.881              0.864              0.894                     0.400
         M2_lgbm_text 0.747       0.301           0.888              0.874              0.902                     0.398
T_finetuned_text_only 0.696       0.285           0.840              0.825              0.855                     0.393
    M3_lgbm_finetuned 0.750       0.302           0.891              0.878              0.902                     0.403

M3 minus M1_lgbm: +1.0 points of hit ra